### Accessing cloud locations using `Cloud Location Finder API`

#### The goal is to:

* Access cloud locations and turn them into a dataset 


## Provider of the cloud location

* CLOUD_PROVIDER_UNSPECIFIED = 0
* CLOUD_PROVIDER_GCP = 1
* CLOUD_PROVIDER_AWS = 2
* CLOUD_PROVIDER_AZURE = 3
* CLOUD_PROVIDER_OCI = 4

## Type of cloud Location

* CLOUD_LOCATION_TYPE_UNSPECIFIED = 0
* CLOUD_LOCATION_TYPE_REGION = 1
* CLOUD_LOCATION_TYPE_ZONE = 2
* CLOUD_LOCATION_TYPE_GDCC_ZONE = 3


In [1]:
import pandas as pd
from google.cloud import locationfinder_v1

client = locationfinder_v1.CloudLocationFinderClient()
parent_value = "projects/datacentre-project/locations/global"

request = locationfinder_v1.ListCloudLocationsRequest(parent=parent_value)             
pager = client.list_cloud_locations(request=request)

provider_map = {
    0: "UNSPECIFIED",
    1: "GCP",
    2: "AWS",
    3: "AZURE",
    4: "OCI"
}

type_map = {
    0: "UNSPECIFIED",
    1: "REGION",
    2: "ZONE",
    3: "GDCC_ZONE"
}

rows = []

for location in pager:
    p_id = int(location.cloud_provider)
    t_id = int(location.cloud_location_type)
    
    cfe = getattr(location, 'carbon_free_energy_percentage', None)
    
    # Extract only the last part of the resource paths to mask your project ID
    clean_location_id = location.name.split('/')[-1] if location.name else None
    clean_containing = location.containing_cloud_location.split('/')[-1] if location.containing_cloud_location else None
    
    row = {
        "display_name": location.display_name,
        "provider": provider_map.get(p_id, f"UNKNOWN({p_id})"),
        "provider_id": p_id,
        "territory_code": location.territory_code,
        "location_type": type_map.get(t_id, f"UNKNOWN({t_id})"),
        "type_id": t_id,
        "carbon_free_energy_percentage": cfe if cfe is not None else None,
        "containing_location_id": clean_containing,  # Cleaned parent ID (e.g., gcp-europe-north2)
        "location_id": clean_location_id             # Cleaned ID (e.g., gcp-europe-north2-a)
    }
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv("../datasets/cloud_locations_dataset.csv", index=False)

print(f"Dataset successfully created with {len(df)} rows and saved to your datasets folder!")

/opt/hostedtoolcache/Python/3.10.20/x64/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


/opt/hostedtoolcache/Python/3.10.20/x64/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.locationfinder_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.locationfinder_v1 past that date.
  warnings.warn(message, FutureWarning)


Dataset successfully created with 587 rows and saved to your datasets folder!
